<a href="https://colab.research.google.com/github/mabelzunce/aprendizaje-profundo-unsam/blob/main/TP2_Atencion_MiniGPT_version_alumnos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# TP2 — Atención y mini-GPT character-level en español

**Materia:** Aprendizaje Profundo — UNSAM  
**Tema:** Self-Attention, Transformer Decoder y modelos de lenguaje autoregresivos  
**Modalidad:** individual
**Entrega:** notebook ejecutado con comentarios y explicaciones delos resultados dentro del mismo notebook. Luego defensa oral del mismo con una presentación de 5-10 minutos.

Este TP construye un modelo de lenguaje pequeño, inspirado en la filosofía de nanoGPT: un Transformer decoder entrenado para predecir el próximo carácter en un corpus chico en español. El objetivo no es obtener un LLM competitivo, sino entender y evaluar los mecanismos internos que hacen funcionar a los LLM modernos.

## Objetivos

Al finalizar el TP deberías poder:

1. Implementar **scaled dot-product attention** y entender el rol de la máscara causal.
2. Implementar una capa de **Multi-Head Self-Attention**.
3. Construir un bloque Transformer decoder con **LayerNorm, atención, FFN y conexiones residuales**.
4. Entrenar un modelo autoregresivo character-level.
5. Evaluar el modelo con métricas cuantitativas y cualitativas: loss, perplexity, muestras generadas y mapas de atención.
6. Analizar críticamente qué aprendió el modelo y qué limitaciones tiene.

## Idea general

En un modelo de lenguaje autoregresivo queremos estimar

$$
P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1}).
$$

Por eso, durante el entrenamiento, el modelo recibe una secuencia de entrada y aprende a predecir el próximo token. Como trabajaremos a nivel carácter, cada token será un carácter del corpus.

> **Importante:** este modelo es un mini-LLM didáctico para estudiar los componentes fundamentales de un Transformer decoder.


In [ ]:

# ============================================================
# 0. Setup
# ============================================================

import math
import os
import random
from dataclasses import dataclass
from typing import Optional, Tuple, List

import numpy as np
import torch
torch.set_num_threads(1)  # evita overhead de multithreading en modelos chicos/CPU
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device: {device}")



## 1. Corpus y tokenización character-level

Usaremos un corpus breve en español. Si existe un archivo `martin_fierro.txt` en el mismo directorio, el notebook lo usará automáticamente. Si no existe, se usa un corpus mínimo incluido en el notebook para que el TP sea reproducible sin descargar nada.

La tokenización character-level queda dada por el notebook. No es el foco del TP. El foco es implementar el Transformer por dentro.


In [ ]:
# ============================================================
# Descarga automática del Martín Fierro (Project Gutenberg)
# Si el archivo ya existe, no se vuelve a descargar.
# ============================================================

import urllib.request

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/14765/pg14765.txt"
LOCAL_PATH = "martin_fierro.txt"

if not os.path.exists(LOCAL_PATH):
    print("Descargando Martín Fierro desde Project Gutenberg...")
    try:
        urllib.request.urlretrieve(GUTENBERG_URL, LOCAL_PATH)
        # Verificación básica
        with open(LOCAL_PATH, "r", encoding="utf-8", errors="replace") as f:
            raw = f.read()
        # El archivo de Gutenberg tiene cabecera/pie en inglés; los eliminamos.
        start = raw.find("MARTIN FIERRO")
        if start == -1:
            start = raw.find("Martín Fierro")
        if start == -1:
            start = 0
        end = raw.rfind("End of the Project Gutenberg")
        if end == -1:
            end = len(raw)
        clean = raw[start:end]
        with open(LOCAL_PATH, "w", encoding="utf-8") as f:
            f.write(clean)
        print(f"✅ Descargado y limpiado: {len(clean):,} caracteres guardados en '{LOCAL_PATH}'.")
    except Exception as e:
        print(f"⚠️  No se pudo descargar ({e}). Se usará el corpus mínimo incluido en el notebook.")
        if os.path.exists(LOCAL_PATH):
            os.remove(LOCAL_PATH)
else:
    with open(LOCAL_PATH, "r", encoding="utf-8", errors="replace") as f:
        _n = len(f.read())
    print(f"✅ '{LOCAL_PATH}' ya existe ({_n:,} caracteres). No se vuelve a descargar.")

In [ ]:
# ============================================================
# 1. Datos
# ============================================================

FALLBACK_TEXT = """
Aquí me pongo a cantar
al compás de la vigüela,
que al hombre que lo desvela
una pena extraordinaria,
como el ave solitaria
con el cantar se consuela.

Pido a los santos del cielo
que ayuden mi pensamiento;
les pido en este momento
que voy a cantar mi historia
me refresquen la memoria
y aclaren mi entendimiento.

Vengan santos milagrosos,
vengan todos en mi ayuda,
que la lengua se me añuda
y se me turba la vista;
pido a mi Dios que me asista
en una ocasión tan ruda.

Yo he conocido esta tierra
en que el paisano vivía
y su ranchito tenía
y sus hijos y mujer;
era una delicia el ver
cómo pasaba sus días.

Entonces, cuando el lucero
brillaba en el cielo santo,
y los gallos con su canto
nos decían que el día llegaba,
a la cocina rumbiaba
el gaucho que era un encanto.

Y sentao junto al fogón
a esperar que venga el día,
al cimarrón le prendía
hasta ponerse rechoncho,
mientras su china dormía
tapadita con su poncho.

"""

def load_corpus(path: str = "martin_fierro.txt") -> str:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        print(f"Corpus cargado desde {path}. Caracteres: {len(text):,}")
        return text
    print("No se encontró martin_fierro.txt. Usando corpus mínimo incluido en el notebook.")
    # Repetimos el texto para tener una cantidad razonable de caracteres.
    return FALLBACK_TEXT * 80

text = load_corpus()
print(text[:500])
print("\nCantidad de caracteres:", len(text))

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

# Tokenización dada: carácter -> índice entero.
def encode(s: str) -> List[int]:
    return [stoi[c] for c in s if c in stoi]

# Decodificación: índice entero -> carácter.
def decode(ids: List[int]) -> str:
    return "".join(itos[int(i)] for i in ids)

print(f"Tamaño del vocabulario: {vocab_size}")
print("Vocabulario:", "".join(chars))

# Dataset como una secuencia larga de enteros.
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")



## 2. Batches autoregresivos

Para cada bloque de longitud `block_size`, la entrada `x` contiene caracteres desde `t` hasta `t + block_size - 1`, y el target `y` contiene los mismos caracteres desplazados una posición hacia adelante.

Ejemplo conceptual:

```text
x = "Aquí me pong"
y = "quí me pongo"
```

El modelo debe aprender a predecir cada próximo carácter.


In [ ]:
# ============================================================
# 2. Batches autoregresivos
# ============================================================

block_size = 32
batch_size = 8

def get_batch(split: str, batch_size: int = batch_size, block_size: int = block_size) -> Tuple[torch.Tensor, torch.Tensor]:
    source = train_data if split == "train" else val_data
    if len(source) <= block_size + 1:
        raise ValueError("El corpus es demasiado chico para el block_size elegido.")
    ix = torch.randint(0, len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x shape:", xb.shape)
print("y shape:", yb.shape)
print("\nEjemplo x:")
print(decode(xb[0].detach().cpu().tolist()))
print("\nEjemplo y:")
print(decode(yb[0].detach().cpu().tolist()))



## 3. Scaled Dot-Product Attention

La atención transforma tres tensores:

- **Q**: queries, lo que cada posición busca.
- **K**: keys, lo que cada posición ofrece para ser comparada.
- **V**: values, la información que se combina después de calcular los pesos.

La operación central es:

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V,
$$

con `M` una máscara causal que impide mirar posiciones futuras.

### Pregunta conceptual

¿Por qué un modelo autoregresivo (por ejemplo, GPT) necesita una máscara causal (masked attention)? ¿Qué información indebida tendría el modelo si no la usáramos?


In [ ]:
# ============================================================
# 3. Scaled Dot-Product Attention
# ============================================================

def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    causal: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Calcula atención escalada.

    Parámetros
    ----------
    q, k, v: tensores de forma (B, T, C) o (..., T, C)
    causal: si True, aplica máscara triangular inferior.

    Retorna
    -------
    out: salida de atención, misma forma temporal que q.
    weights: matriz de pesos de atención.

    TODO:
    1. Calcular scores = Q K^T / sqrt(d_k).
    2. Si causal=True, aplicar máscara triangular inferior.
    3. Aplicar softmax sobre la última dimensión.
    4. Multiplicar los pesos por V.
    """
    # ===== TU CÓDIGO AQUÍ =====
    raise NotImplementedError("Implementar scaled_dot_product_attention")

# Test mínimo de forma y causalidad.
B, T, C = 2, 5, 8
q = torch.randn(B, T, C)
k = torch.randn(B, T, C)
v = torch.randn(B, T, C)
out, weights = scaled_dot_product_attention(q, k, v, causal=True)

assert out.shape == (B, T, C)
assert weights.shape == (B, T, T)
assert torch.allclose(weights[0].triu(1), torch.zeros_like(weights[0].triu(1)), atol=1e-6)
print("Tests básicos OK")


### ✅ Checkpoint 1 — `scaled_dot_product_attention`

> **Antes de continuar**, ejecutá la celda siguiente. Si algún test falla, revisá tu implementación. Las secciones 4, 5, 6, 7 y 8 dependen directamente de que esta función esté correcta.

In [ ]:
# ============================================================
# CHECKPOINT 1: verificación de scaled_dot_product_attention
# No modificar esta celda.
# ============================================================

import traceback

def _run_checkpoint_1():
    passed = 0
    failed = 0

    def check(name, fn):
        nonlocal passed, failed
        try:
            fn()
            print(f"  ✅ {name}")
            passed += 1
        except Exception as e:
            print(f"  ❌ {name}")
            traceback.print_exc()
            failed += 1

    print("── Checkpoint 1: scaled_dot_product_attention ──")

    # Test 1: forma de salida
    def t1():
        B, T, C = 2, 5, 8
        q_ = torch.randn(B, T, C)
        k_ = torch.randn(B, T, C)
        v_ = torch.randn(B, T, C)
        out, w = scaled_dot_product_attention(q_, k_, v_, causal=True)
        assert out.shape == (B, T, C), f"out.shape esperado {(B,T,C)}, obtenido {out.shape}"
        assert w.shape == (B, T, T),   f"weights.shape esperado {(B,T,T)}, obtenido {w.shape}"
    check("Forma de salida (B,T,C) y pesos (B,T,T)", t1)

    # Test 2: causalidad estricta — parte superior = 0
    def t2():
        B, T, C = 1, 6, 8
        q_ = torch.randn(B, T, C)
        k_ = torch.randn(B, T, C)
        v_ = torch.randn(B, T, C)
        _, w = scaled_dot_product_attention(q_, k_, v_, causal=True)
        upper = w[0].triu(1)
        assert torch.allclose(upper, torch.zeros_like(upper), atol=1e-6), \
            f"La máscara causal no es correcta. Máximos sobre la diagonal superior: {upper.max().item():.6f}"
    check("Máscara causal: triángulo superior == 0", t2)

    # Test 3: los pesos suman 1 por fila
    def t3():
        B, T, C = 2, 4, 8
        q_ = torch.randn(B, T, C)
        k_ = torch.randn(B, T, C)
        v_ = torch.randn(B, T, C)
        _, w = scaled_dot_product_attention(q_, k_, v_, causal=True)
        row_sums = w.sum(dim=-1)
        assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5), \
            f"Los pesos no suman 1 por fila. Sumas: {row_sums}"
    check("Pesos de atención suman 1 por fila (softmax correcto)", t3)

    # Test 4: sin máscara causal, todos los pesos > 0
    def t4():
        B, T, C = 1, 4, 8
        q_ = torch.randn(B, T, C)
        k_ = torch.randn(B, T, C)
        v_ = torch.randn(B, T, C)
        _, w = scaled_dot_product_attention(q_, k_, v_, causal=False)
        assert (w > 0).all(), "Con causal=False todos los pesos deberían ser > 0"
    check("Sin máscara causal todos los pesos son positivos", t4)

    # Test 5: escalado correcto — scores deben estar divididos por sqrt(d_k)
    def t5():
        torch.manual_seed(0)
        B, T, C = 1, 3, 64   # d_k grande para que el efecto sea notable
        q_ = torch.ones(B, T, C)
        k_ = torch.ones(B, T, C)
        v_ = torch.ones(B, T, C)
        _, w = scaled_dot_product_attention(q_, k_, v_, causal=False)
        # Con q=k=1 y sin escalar: scores = C = 64 → softmax muy picudo (≈ uniforme en limite opuesto)
        # Con escalado correcto scores = C/sqrt(C) = sqrt(C) = 8
        # Si NO se escala, la distribución colapsará a (casi) uniforme igualmente pero verificamos
        # indirectamente que los pesos son iguales entre sí (Q y K idénticos → scores idénticos)
        # Verificación: todos los pesos de la misma fila son iguales
        diff = w[0].std(dim=-1).max().item()
        assert diff < 1e-5, f"Con Q=K=cte los pesos de cada fila deberían ser iguales. Std máxima: {diff}"
    check("Escalado: con Q=K constante los pesos son uniformes", t5)

    print()
    if failed == 0:
        print(f"🎉 Todos los tests pasaron ({passed}/{passed}). Podés continuar con la sección 4.")
    else:
        print(f"⚠️  {failed} test(s) fallaron. Revisá tu implementación antes de continuar.")

_run_checkpoint_1()

In [ ]:
# Visualización de la máscara causal y de una matriz de atención.
T = 12
q = torch.randn(1, T, 16)
k = torch.randn(1, T, 16)
v = torch.randn(1, T, 16)
_, w = scaled_dot_product_attention(q, k, v, causal=True)

plt.figure(figsize=(5, 4))
plt.imshow(w[0].detach().cpu())
plt.title("Pesos de atención causal")
plt.xlabel("posición atendida")
plt.ylabel("posición que consulta")
plt.colorbar()
plt.show()



## 4. Implementación del Transformer decoder

Ahora construiremos los módulos internos:

1. Una cabeza de atención causal (`Head`).
2. Varias cabezas en paralelo (`MultiHeadAttention`).
3. Una red feed-forward por posición (`FeedForward`).
4. Un bloque Transformer decoder (`Block`).
5. Un mini-GPT character-level (`MiniGPT`).

La arquitectura que usaremos es **decoder-only**, como GPT: atención causal + predicción del próximo token.


In [ ]:
# ============================================================
# 4. Módulos del Transformer decoder
# ============================================================

class Head(nn.Module):
    """Una cabeza de self-attention causal."""
    def __init__(self, n_embd: int, head_size: int, block_size: int, dropout: float):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        B, T, C = x.shape

        # TODO:
        # 1. Proyectar x a k, q, v.
        # 2. Calcular scores q @ k.T / sqrt(head_size).
        # 3. Aplicar máscara causal usando self.tril.
        # 4. Aplicar softmax y dropout.
        # 5. Multiplicar por v.
        # 6. Si return_attn=True, retornar también los pesos.

        # ===== TU CÓDIGO AQUÍ =====
        raise NotImplementedError("Implementar Head.forward")


class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention causal."""
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd debe ser divisible por n_head"
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        # TODO:
        # 1. Ejecutar todas las cabezas en paralelo conceptualmente.
        # 2. Concatenar por la dimensión de canales.
        # 3. Aplicar proyección final y dropout.
        # 4. Si return_attn=True, retornar también un tensor de atención
        #    de forma (B, n_head, T, T).

        # ===== TU CÓDIGO AQUÍ =====
        raise NotImplementedError("Implementar MultiHeadAttention.forward")


class FeedForward(nn.Module):
    """MLP aplicada independientemente a cada posición."""
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    """Bloque Transformer decoder: LN -> MHA -> residual -> LN -> FFN -> residual."""
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        # TODO:
        # Implementar el bloque con conexiones residuales.
        # Recomendación: usar arquitectura pre-norm:
        # x = x + self.sa(self.ln1(x))
        # x = x + self.ffwd(self.ln2(x))
        # Considerar el caso return_attn=True.

        # ===== TU CÓDIGO AQUÍ =====
        raise NotImplementedError("Implementar Block.forward")


### ✅ Checkpoint 2 — `Head`, `MultiHeadAttention`, `Block`

> Ejecutá la celda siguiente antes de continuar con `MiniGPT`. Verifica forma de salida, causalidad y que `return_attn` funcione correctamente.

In [ ]:
# ============================================================
# CHECKPOINT 2: Head, MultiHeadAttention, Block
# No modificar esta celda.
# ============================================================

def _run_checkpoint_2():
    passed = 0
    failed = 0

    def check(name, fn):
        nonlocal passed, failed
        try:
            fn()
            print(f"  ✅ {name}")
            passed += 1
        except Exception as e:
            print(f"  ❌ {name}: {e}")
            failed += 1

    n_embd, n_head, bs, drop = 32, 4, block_size, 0.0
    head_size = n_embd // n_head
    B_, T_ = 2, 10

    print("── Checkpoint 2: módulos del Transformer ──")

    # Test 1: Head — forma de salida
    def t1():
        h = Head(n_embd, head_size, bs, drop)
        x = torch.randn(B_, T_, n_embd)
        out = h(x)
        assert out.shape == (B_, T_, head_size), \
            f"Head: esperado {(B_, T_, head_size)}, obtenido {out.shape}"
    check("Head — forma de salida (B, T, head_size)", t1)

    # Test 2: Head — causalidad (sin dropout)
    def t2():
        h = Head(n_embd, head_size, bs, drop)
        h.eval()
        x = torch.randn(B_, T_, n_embd)
        # Modificamos el primer token y vemos si afecta posiciones anteriores
        x2 = x.clone()
        x2[:, 0, :] += 10.0  # perturbamos token 0
        out1 = h(x)
        out2 = h(x2)
        # La posición 0 puede cambiar (se atiende a sí misma)
        # Las posiciones anteriores a 0 no existen, verificamos que pos 1+ no cambie para tokens [0]
        # Verificación alternativa: pesos de atención en posición (i, j) con j>i deben ser 0
        _, attn = h(x, return_attn=True)
        upper = attn[0].triu(1)
        assert torch.allclose(upper, torch.zeros_like(upper), atol=1e-5), \
            "Head: los pesos de atención violan la causalidad (triángulo superior != 0)"
    check("Head — causalidad respetada", t2)

    # Test 3: Head — return_attn devuelve tupla (out, weights)
    def t3():
        h = Head(n_embd, head_size, bs, drop)
        x = torch.randn(B_, T_, n_embd)
        result = h(x, return_attn=True)
        assert isinstance(result, tuple) and len(result) == 2, \
            "Head: return_attn=True debe devolver (out, weights)"
        out, w = result
        assert out.shape == (B_, T_, head_size)
        assert w.shape == (B_, T_, T_)
    check("Head — return_attn=True devuelve (out, weights)", t3)

    # Test 4: MultiHeadAttention — forma de salida
    def t4():
        mha = MultiHeadAttention(n_embd, n_head, bs, drop)
        x = torch.randn(B_, T_, n_embd)
        out = mha(x)
        assert out.shape == (B_, T_, n_embd), \
            f"MHA: esperado {(B_, T_, n_embd)}, obtenido {out.shape}"
    check("MultiHeadAttention — forma de salida (B, T, n_embd)", t4)

    # Test 5: MultiHeadAttention — return_attn devuelve (out, attn_stack)
    def t5():
        mha = MultiHeadAttention(n_embd, n_head, bs, drop)
        x = torch.randn(B_, T_, n_embd)
        result = mha(x, return_attn=True)
        assert isinstance(result, tuple) and len(result) == 2, \
            "MHA: return_attn=True debe devolver (out, attn_stack)"
        out, attn_stack = result
        assert out.shape == (B_, T_, n_embd)
        assert attn_stack.shape == (B_, n_head, T_, T_), \
            f"MHA attn_stack: esperado {(B_, n_head, T_, T_)}, obtenido {attn_stack.shape}"
    check("MultiHeadAttention — return_attn devuelve (B, n_head, T, T)", t5)

    # Test 6: Block — forma de salida (residual mantiene dimensión)
    def t6():
        blk = Block(n_embd, n_head, bs, drop)
        x = torch.randn(B_, T_, n_embd)
        out = blk(x)
        assert out.shape == (B_, T_, n_embd), \
            f"Block: esperado {(B_, T_, n_embd)}, obtenido {out.shape}"
    check("Block — conexión residual mantiene la forma (B, T, n_embd)", t6)

    # Test 7: Block — return_attn
    def t7():
        blk = Block(n_embd, n_head, bs, drop)
        x = torch.randn(B_, T_, n_embd)
        result = blk(x, return_attn=True)
        assert isinstance(result, tuple) and len(result) == 2, \
            "Block: return_attn=True debe devolver (out, attn)"
        out, attn = result
        assert out.shape == (B_, T_, n_embd)
    check("Block — return_attn=True devuelve (out, attn)", t7)

    print()
    if failed == 0:
        print(f"🎉 Todos los tests pasaron ({passed}/{passed}). Podés continuar con MiniGPT.")
    else:
        print(f"⚠️  {failed} test(s) fallaron. Revisá tu implementación antes de continuar.")

_run_checkpoint_2()


## 5. MiniGPT character-level

El modelo completo suma:

- embeddings de tokens,
- embeddings posicionales aprendidos,
- varios bloques Transformer decoder,
- una capa final que produce logits sobre el vocabulario.

Entrenaremos con `CrossEntropyLoss`, equivalente a maximizar la probabilidad del próximo carácter correcto.


In [ ]:
# ============================================================
# 5. MiniGPT character-level
# ============================================================

@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 32
    n_embd: int = 32
    n_head: int = 4
    n_layer: int = 2
    dropout: float = 0.1
    batch_size: int = 8


class MiniGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.ModuleList([
            Block(config.n_embd, config.n_head, config.block_size, config.dropout)
            for _ in range(config.n_layer)
        ])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ):
        B, T = idx.shape
        if T > self.config.block_size:
            raise ValueError("La longitud de contexto supera block_size")

        # TODO:
        # 1. Obtener embeddings de tokens.
        # 2. Obtener embeddings posicionales.
        # 3. Sumar ambos.
        # 4. Pasar por los bloques Transformer.
        # 5. Aplicar LayerNorm final y lm_head.
        # 6. Si targets no es None, calcular cross entropy.
        # 7. Si return_attn=True, retornar también los mapas de atención.

        # ===== TU CÓDIGO AQUÍ =====
        raise NotImplementedError("Implementar MiniGPT.forward")

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
    ) -> torch.Tensor:
        # TODO:
        # Implementar generación autoregresiva:
        # 1. Recortar el contexto a block_size.
        # 2. Ejecutar el modelo.
        # 3. Tomar los logits de la última posición.
        # 4. Aplicar temperature y opcionalmente top_k.
        # 5. Muestrear el próximo token.
        # 6. Concatenarlo al contexto.

        # ===== TU CÓDIGO AQUÍ =====
        raise NotImplementedError("Implementar MiniGPT.generate")


config = GPTConfig(
    vocab_size=vocab_size,
    block_size=32,
    n_embd=32,
    n_head=4,
    n_layer=1,
    dropout=0.1,
    batch_size=8,
)

model = MiniGPT(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nCantidad de parámetros: {n_params:,}")

xb, yb = get_batch("train", config.batch_size, config.block_size)
logits, loss = model(xb, yb)
print("logits shape:", logits.shape)
print("loss inicial:", float(loss.detach().cpu()))



## 6. Entrenamiento

Entrenaremos pocos pasos para que el TP corra en CPU. En GPU se puede aumentar `max_iters`.

La métrica principal será la **negative log-likelihood promedio**, reportada como `cross entropy loss`. También calcularemos **perplexity**:

$$
\operatorname{PPL} = e^{\operatorname{loss}}.
$$

Menor loss/perplexity indica que el modelo asigna mayor probabilidad al próximo carácter correcto.


In [ ]:
# ============================================================
# 6. Entrenamiento
# ============================================================

@torch.no_grad()
def estimate_loss(model: nn.Module, eval_iters: int = 20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            X, Y = get_batch(split, config.batch_size, config.block_size)
            _, loss = model(X, Y)
            losses.append(loss.item())
        out[split] = float(np.mean(losses))
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

max_iters = 3000 if device != "cpu" else 1000
eval_interval = 200 if device != "cpu" else 100
history = []

for it in range(max_iters + 1):
    if it % eval_interval == 0:
        losses = estimate_loss(model, eval_iters=20)
        history.append({"iter": it, **losses})
        print(
            f"iter {it:4d} | "
            f"train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f} | "
            f"val ppl {math.exp(losses['val']):.2f}"
        )

    xb, yb = get_batch("train", config.batch_size, config.block_size)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()


In [ ]:
# Curvas de entrenamiento y validación.
iters = [h["iter"] for h in history]
train_losses = [h["train"] for h in history]
val_losses = [h["val"] for h in history]

plt.figure(figsize=(6, 4))
plt.plot(iters, train_losses, marker="o", label="train")
plt.plot(iters, val_losses, marker="o", label="val")
plt.xlabel("Iteración")
plt.ylabel("Cross-entropy loss")
plt.title("Curva de entrenamiento")
plt.legend()
plt.grid(True)
plt.show()



## 7. Generación de texto

Ahora generamos texto autoregresivamente. No esperamos resultados perfectos: el corpus es chico, el modelo es chico y el entrenamiento es breve. Aun así, deberíamos observar que el modelo empieza a aprender estructura local: espacios, saltos de línea, combinaciones frecuentes de letras y algunas palabras.

### Para discutir

- ¿Qué cambia al modificar `temperature`?
- ¿Qué cambia al modificar `top_k`?
- ¿El texto generado es gramatical? ¿Tiene coherencia global?
- ¿Qué necesitaríamos para acercarnos a un LLM real?


In [ ]:
# ============================================================
# 7. Generación
# ============================================================

prompt = "Aquí "
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=300,
    temperature=0.9,
    top_k=20,
)

print(decode(generated[0].detach().cpu().tolist()))



## 8. Visualización de mapas de atención

Inspeccionaremos los pesos de atención de una capa y una cabeza. Esta visualización no debe sobreinterpretarse: una cabeza de atención no es una explicación completa del modelo. Sin embargo, sirve para ver si la máscara causal se respeta y cómo distintas posiciones distribuyen su atención sobre el contexto pasado.


In [ ]:
# ============================================================
# 8. Mapas de atención del modelo entrenado
# ============================================================

model.eval()
prompt = "Aquí me pongo"
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

with torch.no_grad():
    logits, loss, attn_maps = model(idx, return_attn=True)

# Elegimos última capa y primera cabeza.
layer_id = -1
head_id = 0
attn = attn_maps[layer_id][0, head_id].detach().cpu()
labels = list(decode(idx[0].detach().cpu().tolist()))

plt.figure(figsize=(7, 6))
plt.imshow(attn)
plt.title(f"Atención — capa {len(attn_maps) + layer_id if layer_id < 0 else layer_id}, cabeza {head_id}")
plt.xticks(range(len(labels)), labels, rotation=90)
plt.yticks(range(len(labels)), labels)
plt.xlabel("posición atendida")
plt.ylabel("posición que consulta")
plt.colorbar()
plt.tight_layout()
plt.show()

model.train()



## 9. Experimentos de evaluación / ablation study

Para que el TP no sea sólo implementación, deberán comparar al menos **dos configuraciones** y reportar resultados.

Sugerencias:

1. `n_head = 1` vs `n_head = 4`.
2. `block_size = 32` vs `block_size = 64`.
3. `n_layer = 1` vs `n_layer = 2`.
4. `dropout = 0.0` vs `dropout = 0.2`.
5. Entrenar durante pocas vs más iteraciones.

Para cada experimento, reportar:

- número de parámetros,
- train loss,
- validation loss,
- validation perplexity,
- una muestra generada,
- una conclusión breve.

Completar la tabla al final.


In [ ]:
# ============================================================
# 9. Utilidad para entrenar variantes pequeñas
# ============================================================

def train_small_model(cfg: GPTConfig, max_iters: int = 80, eval_iters: int = 10, lr: float = 3e-4):
    m = MiniGPT(cfg).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr)
    hist = []

    for it in range(max_iters + 1):
        if it % max(1, max_iters // 4) == 0:
            losses = estimate_loss_for_config(m, cfg, eval_iters=eval_iters)
            hist.append({"iter": it, **losses})

        xb, yb = get_batch("train", cfg.batch_size, cfg.block_size)
        _, loss = m(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
        opt.step()

    final_losses = estimate_loss_for_config(m, cfg, eval_iters=eval_iters)
    return m, hist, final_losses

@torch.no_grad()
def estimate_loss_for_config(m: nn.Module, cfg: GPTConfig, eval_iters: int = 10):
    m.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            X, Y = get_batch(split, cfg.batch_size, cfg.block_size)
            _, loss = m(X, Y)
            losses.append(loss.item())
        out[split] = float(np.mean(losses))
    m.train()
    return out

# Ejemplo de comparación rápida. Podés cambiar estas configuraciones.
configs = {
    "1_head": GPTConfig(vocab_size=vocab_size, block_size=32, n_embd=32, n_head=1, n_layer=1, dropout=0.1, batch_size=8),
    "4_heads": GPTConfig(vocab_size=vocab_size, block_size=32, n_embd=32, n_head=4, n_layer=1, dropout=0.1, batch_size=8),
}

results = []
trained_variants = {}
for name, cfg in configs.items():
    print(f"\nEntrenando variante: {name}")
    m, hist, losses = train_small_model(cfg, max_iters=8 if device == "cpu" else 50, eval_iters=2)
    n_params = sum(p.numel() for p in m.parameters())
    results.append({
        "modelo": name,
        "params": n_params,
        "train_loss": losses["train"],
        "val_loss": losses["val"],
        "val_ppl": math.exp(losses["val"]),
    })
    trained_variants[name] = m

results



## 10. Informe dentro del notebook

Completar esta sección antes de entregar.

### 10.1 Implementación

Explicar brevemente qué hace cada módulo:

- `scaled_dot_product_attention`
- `Head`
- `MultiHeadAttention`
- `FeedForward`
- `Block`
- `MiniGPT`

### 10.2 Resultados cuantitativos

Completar la tabla:

| Modelo | Parámetros | Train loss | Val loss | Val perplexity | Comentario |
|---|---:|---:|---:|---:|---|
| baseline | | | | | |
| variante 1 | | | | | |
| variante 2 | | | | | |

### 10.3 Resultados cualitativos

Pegar 2 o 3 muestras generadas y comentar:

- ¿Aparecen palabras válidas?
- ¿Respeta espacios y saltos de línea?
- ¿Hay coherencia local?
- ¿Hay coherencia global?

### 10.4 Atención

Incluir al menos un mapa de atención y responder:

- ¿Se respeta la causalidad?
- ¿Qué patrones aparecen?
- ¿Todas las cabezas parecen hacer lo mismo?

### 10.5 Conclusión

Responder en 10–15 líneas:

1. ¿Qué aprendió el modelo?
2. ¿Cuáles son sus limitaciones principales?
3. ¿Qué diferencias hay entre este mini-GPT y un LLM moderno?
4. ¿Qué cambiarías si tuvieras más datos y GPU?



## Rúbrica sugerida de corrección

| Ítem | Peso |
|---|---:|
| Implementación correcta de atención causal y tests básicos | 20% |
| Implementación correcta de Multi-Head Attention, FFN, residuales y LayerNorm | 25% |
| Entrenamiento reproducible con curvas y métricas | 15% |
| Evaluación: loss, perplexity, muestras y ablations | 20% |
| Análisis de mapas de atención | 10% |
| Claridad del informe y conclusiones críticas | 10% |

### Requisitos mínimos para aprobar

- El notebook debe correr de punta a punta.
- La máscara causal debe estar correctamente implementada.
- Debe haber al menos una comparación experimental.
- Debe haber una conclusión propia, no sólo outputs de código.

### Extensiones opcionales

- Usar un corpus más grande en español.
- Agregar scheduler de learning rate.
- Guardar y cargar checkpoints.
- Comparar tokenización character-level vs subword usando una librería externa.
- Visualizar varias cabezas y capas de atención.



## Referencias sugeridas

- Vaswani et al. (2017), *Attention Is All You Need*.
- Andrej Karpathy, nanoGPT: repositorio didáctico para entrenar modelos GPT pequeños.
- Material de clase sobre atención, Transformers y modelos autoregresivos.
